In [3]:
# Imports
%matplotlib inline
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import os
from sklearn.linear_model import LogisticRegression


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
from src.config_presets.tools.get_config import get_config
from src.constants import PATIENT_ID_COL_NAME, PATIENT_ID_LENGTHS_DICT, EARLY_NTCP_TIMEPOINTS, LATE_NTCP_TIMEPOINTS

config = get_config('Trial32_Config')

PATIENT_ID_LENGTH = PATIENT_ID_LENGTHS_DICT[config['data']['source']]


src\config_presets\Base_config.yaml
src\config_presets\Trial32_Config.yaml


In [6]:
df_features_dir = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\DL_NTCP_Multitox\datasets\MT_dataset\stratified_sampling_test_542.csv"
#df_features_dir = config['paths']['csv']

df_features = pd.read_csv(df_features_dir, delimiter=';')
df_features[PATIENT_ID_COL_NAME] = df_features[PATIENT_ID_COL_NAME].astype(str).str.zfill(PATIENT_ID_LENGTH)


In [10]:
DL_trial_dir = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_results\Trial_32"

# get the folders in the directory (i.e. the folder per trial)
folder_names = os.listdir(DL_trial_dir)
folder_names = [folder for folder in folder_names if os.path.isdir(os.path.join(DL_trial_dir, folder))]


for k_fold_idx, folder_name in enumerate(folder_names):
    k_fold_folder_dir = os.path.join(DL_trial_dir, folder_name)
    config['general']['resultsCurrentDirectory'] = k_fold_folder_dir
    
    print(k_fold_folder_dir)
    model_preds_df = pd.read_csv(os.path.join(k_fold_folder_dir, "model_predictions.csv"), delimiter=";")

    # check if this df contains the test predictions as well, if not, then also load in the test_set_predictions.csv
    if "test" not in model_preds_df["Mode"].values:
        test_preds_df = pd.read_csv(os.path.join(k_fold_folder_dir, "test_set_predictions.csv"), delimiter=";")
        model_preds_df = pd.concat([model_preds_df, test_preds_df], axis=0)

    model_preds_df[PATIENT_ID_COL_NAME] = model_preds_df[PATIENT_ID_COL_NAME].astype(str).str.zfill(PATIENT_ID_LENGTH)

    train_patient_IDs = model_preds_df[model_preds_df["Mode"] == "train"][PATIENT_ID_COL_NAME].values
    val_patient_IDs = model_preds_df[model_preds_df["Mode"] == "val"][PATIENT_ID_COL_NAME].values
    test_patient_IDs = model_preds_df[model_preds_df["Mode"] == "test"][PATIENT_ID_COL_NAME].values

    run_logistic_regression_posthoc(config, df_features, train_patient_IDs=train_patient_IDs, val_patient_IDs=val_patient_IDs, test_patient_IDs=test_patient_IDs)

    

    
    

\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_results\Trial_32\KFold1
all_lr_predictions.csv
\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_results\Trial_32\KFold1\all_LR_test_metrics.csv
\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_results\Trial_32\KFold2
all_lr_predictions.csv
\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_results\Trial_32\KFold2\all_LR_test_metrics.csv
\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_results\Trial_32\KFold3
all_lr_predictions.csv
\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_results\Trial_32\KFold3\all_LR_test_metrics.csv
\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_results\Trial_32\KFold4
all_lr_predictions.csv
\\zkh\appdata\RTD

In [ ]:
def refit_CITOR_model(config, df_train, endpoint = "Xerostomia_M06", CITOR_model_name = 'xerostomia_late'):
    """
    A function that refits the CITOR model to a training set
    """
    
    if len(config['CITOR'][CITOR_model_name]['models'].keys()) > 1:
        feature_dict = {'model1':{}, 'model2':{}}
        submodel_names = ['model1', 'model2']

        #all_feature_columns = list(set()) 
    else:
        feature_dict = {'model1':{}}
        submodel_names = ['model1']   

    # refit (all of the sub-) models
    for submodel_names in submodel_names:
        # select the columns per submodel
        feature_column_names = np.array(config['CITOR'][CITOR_model_name]['models'][submodel_names]['features'])

        X_train = df_train[feature_column_names]
        y_train = df_train[endpoint]

        # make and fit the (sub)model
        sub_model = LogisticRegression(random_state=config['general']['seed'], C=1e10, max_iter = 1000)

        # Messy things to handle any missing data
        valid_indices = y_train != -1
        X_train = X_train[valid_indices]
        y_train = y_train[valid_indices]
        
        # fit the model
        sub_model.fit(X_train, y_train)

        # save the submodel in the feature_dict
        feature_dict[submodel_names]['features'] = feature_column_names
        feature_dict[submodel_names]['coef'] = sub_model.coef_[0]
        feature_dict[submodel_names]['intercept'] = sub_model.intercept_[0]

        # print(f"submodel {submodel_names} has been fitted")
        # sub_model_dict = {f: c for f, c in zip(feature_column_names, sub_model.coef_[0])}
        # print(sub_model_dict)
        # print(feature_dict[submodel_names]['intercept'])


    # combine the submodels
    if len(feature_dict.keys()) > 1:
        model1 = list(feature_dict.keys())[0]
        model2 = list(feature_dict.keys())[1]
        feature_names = []
        coef = []
        feature_names.append('intercept')
        coef.append(.5*(feature_dict[model1]['intercept'] + feature_dict[model2]['intercept']))
        
        features1 = feature_dict[model1]['features']
        features2 = feature_dict[model2]['features']
        
        overlap =  list(set(features1) & set(features2))
        remainder_m1 = list(set(features1) - set(features2))
        remainder_m2 = list(set(features2) - set(features1))

        
        for i in range(len(remainder_m1)):
            feature_names.append(remainder_m1[i])
            coef.append(.5*(feature_dict[model1]['coef'][np.where(features1 == remainder_m1[i])])[0])         
        for i in range(len(remainder_m2)):
            feature_names.append(remainder_m2[i])
            coef.append(.5*(feature_dict[model2]['coef'][np.where(features2 == remainder_m2[i])])[0])    
        for i in range(len(overlap)):
            feature_names.append(overlap[i])
            coef.append(.5*(feature_dict[model1]['coef'][np.where(features1 == overlap[i])]+feature_dict[model2]['coef'][np.where(features2 == overlap[i])])[0])     
   
    else:
        feature_names = ['intercept'] + list(feature_column_names)
        coef =[sub_model.intercept_[0]] + list(sub_model.coef_[0]) 


    # print(features)
    # features_coeff_dict = {k: v for k, v in zip(features, coef)}
    # print(features_coeff_dict)
    # print(coef[1:])
    
    model = LogisticRegression(random_state=config['general']['seed'], C=1e10, max_iter = 1000, multi_class='ovr')
    model.intercept_ = np.array([coef[0]])
    model.coef_ = np.array([coef[1:]])

    return model, feature_names
    




In [ ]:
from src.utils.saving.saving_predictions import concatenate_predictions, save_predictions


def determine_CITOR_model_name(endpoint_name):
    endpoint_name = endpoint_name.lower()

    toxicity_name = "_".join(endpoint_name.split("_")[:-1])
    if toxicity_name == 'sticky':
        toxicity_name = 'sticky_saliva'

    timepoint = endpoint_name.split("_")[-1]

    CITOR_name = toxicity_name
    if timepoint in EARLY_NTCP_TIMEPOINTS:
        CITOR_name += "_early"
    else:
        CITOR_name += "_late"

    return CITOR_name




from src.evaluation.total_evaluation import total_evaluation_current_fold
from src.evaluation.get_visualisations import get_visualizations


def run_logistic_regression_posthoc(config, df_features, train_patient_IDs, val_patient_IDs, test_patient_IDs=None):
    # get the train and val patients' data
    df_train = df_features[df_features[PATIENT_ID_COL_NAME].isin(train_patient_IDs)]
    df_val = df_features[df_features[PATIENT_ID_COL_NAME].isin(val_patient_IDs)]
    assert len(df_train) == len(train_patient_IDs)
    assert len(df_val) == len(val_patient_IDs)

    # get the test patients' data (if you want that)
    if test_patient_IDs is not None:
        df_test = df_features[df_features[PATIENT_ID_COL_NAME].isin(test_patient_IDs)]
        assert len(df_test) == len(test_patient_IDs)
    else:
        df_test = None

    # print(df_train.shape)
    # print(df_val.shape)
    # print(df_test.shape)

    # places to store the results
    train_y_pred_dict, train_y_true_dict = dict(), dict()
    val_y_pred_dict, val_y_true_dict = dict(), dict()
    test_y_pred_dict, test_y_true_dict = dict(), dict()
    
    endpoint_list = config['columns']['labels']

    all_preds_dict = dict()
    all_targets_dict = dict()
    

    for idx, endpoint in enumerate(endpoint_list):
        #print(f"fitting models for {endpoint}")

        # get the features and labels

        
        #X_train = df_train.loc[:, ]
        CITOR_model_name = determine_CITOR_model_name(endpoint)

        model, feature_column_names = refit_CITOR_model(config, df_train, endpoint = endpoint, CITOR_model_name = CITOR_model_name)

        #print("\n\n")
                         # endpoint, train_patient_IDs, val_patient_IDs, test_patient_IDs)

        # get the predictions
        df_train_X = df_train.loc[:, feature_column_names].values
        df_val_X = df_val.loc[:, feature_column_names].values

        train_patient_IDs = list(df_train[PATIENT_ID_COL_NAME].values)
        val_patient_IDs = list(df_val[PATIENT_ID_COL_NAME].values)

        train_preds = model.predict_proba(df_train_X)[:,1]
        val_preds = model.predict_proba(df_val_X)[:,1]

        if test_patient_IDs is not None:
            df_test_X = df_test.loc[:, feature_column_names].values
            test_patient_IDs = list(df_test[PATIENT_ID_COL_NAME].values)
            test_preds = model.predict_proba(df_test_X)[:,1]
            #print(df_test.shape)
            #print(test_preds.shape)
        else:
            df_test_X = None
            test_patient_IDs = []
            test_preds = None


        #preds = model.predict_proba(df_X)[:,1]

        # store the predictions
        all_preds_dict[endpoint] = np.concatenate([train_preds, val_preds, test_preds])
        all_targets_dict[endpoint] = np.concatenate([df_train[endpoint].values, df_val[endpoint].values, df_test[endpoint].values])

    all_patientIDs_list = train_patient_IDs + val_patient_IDs + test_patient_IDs
    mode_list = ["train"] * len(train_patient_IDs) + ["val"] * len(val_patient_IDs) + ["test"] * len(test_patient_IDs)

    
    # save all the predictions into one csv file
    save_predictions(config, all_patientIDs_list, all_preds_dict, all_targets_dict, mode_list, filename="all_lr_predictions.csv")
    #print("saved all predictions")

    pred_csv_dir = os.path.join(config['general']['resultsCurrentDirectory'], "all_lr_predictions.csv")
    total_evaluation_current_fold(config, sets = ['test'], pred_csv_dir = pred_csv_dir, lr=True)

    get_visualizations(config, pred_csv_dir, lr=True)


    #print(all_preds_dict[endpoint].shape)




    
    # get the features and labels

    
    
    



In [11]:
# do the ensembling of test set CITOR preds

fold_test_set_preds_list = []
test_patientIDs_list_fold1 = None


for k_fold_idx, folder_name in enumerate(folder_names):
    k_fold_folder_dir = os.path.join(DL_trial_dir, folder_name)
    config['general']['resultsCurrentDirectory'] = k_fold_folder_dir
    
    #print(k_fold_folder_dir)
    #model_preds_df = pd.read_csv(os.path.join(k_fold_folder_dir, "model_predictions.csv"), delimiter=";")
    df_fold_preds = pd.read_csv(os.path.join(k_fold_folder_dir, "all_lr_predictions.csv"), delimiter=";")
    df_fold_preds[PATIENT_ID_COL_NAME] = df_fold_preds[PATIENT_ID_COL_NAME].astype(str).str.zfill(PATIENT_ID_LENGTH)
    df_test_preds = df_fold_preds[df_fold_preds["Mode"] == "test"]

    fold_test_set_preds_list.append(df_test_preds.copy())
    test_patientIDs_list = list(df_test_preds[PATIENT_ID_COL_NAME].values)

    if test_patientIDs_list_fold1 == None:
        test_patientIDs_list_fold1 = test_patientIDs_list
    else:
        assert test_patientIDs_list_fold1 == test_patientIDs_list, 'Patient IDs are not the same for all folds'





In [12]:
# Concatenate all dataframes in the list
all_test_preds_df = pd.concat(fold_test_set_preds_list)

# Group by PatientID and Mode, then calculate the mean for each group
ensemble_test_preds_df = all_test_preds_df.groupby(['PatientID', 'Mode']).mean().reset_index()

print(ensemble_test_preds_df.head())

  PatientID  Mode  Aspiration_M06_pred  Aspiration_M06_true  \
0   0052277  test             0.041892                  0.0   
1   0066593  test             0.208485                  0.0   
2   0092560  test             0.060007                  0.0   
3   0163517  test             0.028966                  0.0   
4   0213240  test             0.053075                  0.0   

   Dysphagia_M06_pred  Dysphagia_M06_true  Sticky_M06_pred  Sticky_M06_true  \
0            0.548542                -1.0         0.310419              0.0   
1            0.290522                 0.0         0.311773              0.0   
2            0.153465                 0.0         0.273239              0.0   
3            0.028404                 0.0         0.134322              0.0   
4            0.359584                 0.0         0.270358              1.0   

   Taste_M06_pred  Taste_M06_true  Xerostomia_M06_pred  Xerostomia_M06_true  
0        0.381983             1.0             0.449142              

In [ ]:
config['general']['resultsCurrentDirectory'] = DL_trial_dir
LR_ensemble_preds_dir = os.path.join(DL_trial_dir, "LR_ensemble_preds.csv")
ensemble_test_preds_df.to_csv(LR_ensemble_preds_dir, sep=";", index=False)
    #print("saved all predictions")s

pred_csv_dir = os.path.join(config['general']['resultsCurrentDirectory'], "LR_ensemble_preds.csv")
total_evaluation_current_fold(config, sets = ['test'], pred_csv_dir = pred_csv_dir, lr=True)

get_visualizations(config, sets = ['test'], pred_csv_dir=pred_csv_dir, lr=True)

In [14]:
fold_test_set_preds_list[0]

,PatientID,Mode,Aspiration_M06_pred,Aspiration_M06_true,Dysphagia_M06_pred,Dysphagia_M06_true,Sticky_M06_pred,Sticky_M06_true,Taste_M06_pred,Taste_M06_true,Xerostomia_M06_pred,Xerostomia_M06_true
864,0052277,test,0.043941,0.0,0.590871,-1.0,0.302077,0.0,0.370550,1.0,0.450453,0.0
865,0066593,test,0.212398,0.0,0.284689,0.0,0.305819,0.0,0.302121,0.0,0.448124,0.0
866,0092560,test,0.060298,0.0,0.154393,0.0,0.270907,0.0,0.194388,0.0,0.363655,0.0
867,0163517,test,0.031413,0.0,0.028700,0.0,0.139046,0.0,0.122363,0.0,0.100641,0.0
868,0213240,test,0.054178,0.0,0.349513,0.0,0.269068,1.0,0.516585,0.0,0.575833,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1075,9586209,test,0.063272,0.0,0.743747,1.0,0.345612,0.0,0.494995,0.0,0.566195,1.0
1076,9597467,test,0.060810,0.0,0.300692,1.0,0.328979,0.0,0.386897,1.0,0.520961,1.0
1077,9653665,test,0.056165,0.0,0.315282,0.0,0.278133,1.0,0.516059,1.0,0.377260,1.0
1078,9715913,test,0.053756,1.0,0.378441,1.0,0.491413,1.0,0.513193,1.0,0.665724,1.0


In [153]:
fold_test_set_preds_list[1]

,PatientID,Mode,Aspiration_M06_pred,Aspiration_M06_true,Dysphagia_M06_pred,Dysphagia_M06_true,Sticky_M06_pred,Sticky_M06_true,Taste_M06_pred,Taste_M06_true,Xerostomia_M06_pred,Xerostomia_M06_true
864,0052277,test,0.034449,0.0,0.525764,-1.0,0.361716,0.0,0.300100,1.0,0.449870,0.0
865,0066593,test,0.200972,0.0,0.290623,0.0,0.363599,0.0,0.261440,0.0,0.446676,0.0
866,0092560,test,0.044466,0.0,0.150508,0.0,0.312975,0.0,0.185923,0.0,0.361495,0.0
867,0163517,test,0.026319,0.0,0.030294,0.0,0.134143,0.0,0.140691,0.0,0.098935,0.0
868,0213240,test,0.040780,0.0,0.375947,0.0,0.309217,1.0,0.386973,0.0,0.572641,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1075,9586209,test,0.046234,0.0,0.702572,1.0,0.429247,0.0,0.449941,0.0,0.567746,1.0
1076,9597467,test,0.044771,0.0,0.320176,1.0,0.402854,0.0,0.404096,1.0,0.521549,1.0
1077,9653665,test,0.041984,0.0,0.343202,0.0,0.322235,1.0,0.410109,1.0,0.374915,1.0
1078,9715913,test,0.040524,1.0,0.396427,1.0,0.512719,1.0,0.399627,1.0,0.664008,1.0
